# Classical Gram-Schmidt to QR

Chapter 4: QR Decomposition and Numerical Rank · *Mathematics for AI and Machine Learning* by Fuheng Wu

Compute Q and R for a 2x2 matrix and verify A=QR.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-4/gs-qr/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.set_printoptions(precision=4)

# The demo's default 2x2 matrix
A = torch.tensor([[1.6, 0.7],
                  [0.9, 1.9]], dtype=torch.float64)


def classical_gram_schmidt(A):
    """A = Q R with orthonormal Q and upper-triangular R (classical GS)."""
    m, n = A.shape
    Q = torch.zeros_like(A)
    R = torch.zeros(n, n, dtype=A.dtype)
    for j in range(n):
        v = A[:, j].clone()
        for i in range(j):
            R[i, j] = Q[:, i] @ A[:, j]   # coefficient on the earlier direction
            v -= R[i, j] * Q[:, i]       # remove that component
        R[j, j] = v.norm()
        Q[:, j] = v / R[j, j]
    return Q, R


Q, R = classical_gram_schmidt(A)
print("Q =\n", Q)
print("R =\n", R)
print("A == Q R      :", torch.allclose(Q @ R, A))
print("Q^T Q == I    :", torch.allclose(Q.T @ Q, torch.eye(2, dtype=A.dtype)))
print("|det A| == prod diag R :", abs(torch.linalg.det(A)).item(), R.diagonal().prod().item())

# ── torch.linalg.qr may flip signs (Householder), but spans the same columns ─
Q2, R2 = torch.linalg.qr(A)
signs = torch.sign(R2.diagonal())
print("\nmatches torch.linalg.qr after fixing signs:",
      torch.allclose(Q2 * signs, Q) and torch.allclose(signs[:, None] * R2, R))

Q =
 tensor([[ 0.8716, -0.4903],
        [ 0.4903,  0.8716]], dtype=torch.float64)
R =
 tensor([[1.8358, 1.5416],
        [0.0000, 1.3128]], dtype=torch.float64)
A == Q R      : True
Q^T Q == I    : True
|det A| == prod diag R : 2.41 2.41

matches torch.linalg.qr after fixing signs: True
